# NOTEBOOK 01: DATA PREPARATION & FEATURE ENGINEERING
## HỆ THỐNG HỖ TRỢ HỌC TẬP LỚP 12 (CẢNH BÁO NGUY CƠ & WHAT-IF SIMULATOR)

**Bộ dữ liệu:** `StudentPerformanceFactors.csv` (Kaggle)  
**Phạm vi áp dụng:** Học sinh lớp 12 Việt Nam (1 kỳ học)  

### Mục tiêu tổng quát:
1. **Nạp & Làm sạch:** Nạp dữ liệu thô, sửa lỗi bất thường ngoại lai (`Exam_Score = 101` -> 100), kiểm tra toàn vẹn bản ghi.
2. **Quy chuẩn thang điểm 10:** Chuyển đổi toàn bộ thang điểm thi (`Exam_Score`) và điểm kỳ trước (`Previous_Scores`) từ thang 100 về thang điểm 10 chuẩn Việt Nam.
3. **Giới hạn sinh lý:** Clip thời lượng giấc ngủ `Sleep_Hours` trong khoảng [4, 10] giờ/đêm.
4. **Xử lý dữ liệu khuyết thiếu (Missing Imputation):** Điền giá trị Mode (phổ biến nhất) cho 3 cột phân loại có missing.
5. **Đặc trưng tương tác học tập (Feature Engineering):** Xây dựng `effective_study_index = Hours_Studied * (Attendance / 100.0)` phản ánh tích lũy thời gian học thực tế.
6. **Thiết lập nhãn mục tiêu Tầng 1 (`at_risk`):** Gán `at_risk = 1` khi `Exam_Score < 6.5` (học sinh không đạt chuẩn loại Khá THPT, cần được giáo viên/phụ huynh can thiệp sớm).
7. **Kiểm định chất lượng nghiêm ngặt (Quality Gate):** Toàn bộ điều kiện biên được kiểm tra bằng các câu lệnh `assert` trước khi lưu bảng master.

---
## 📦 EPIC 1: DATA ENGINEERING & PREPARATION
### 🔹 Task 1.1: Nạp dữ liệu, Làm sạch, Đồng bộ thang điểm 10 & Tạo nhãn mục tiêu `at_risk`

**Các quy tắc kỹ thuật áp dụng:**
- **Không rò rỉ dữ liệu (Anti-Leakage):** Cột mục tiêu `Exam_Score` chỉ dùng để tạo nhãn `at_risk` và phục vụ Tầng 2 (hồi quy). Khi huấn luyện Tầng 1 (phân loại), `Exam_Score` sẽ bị loại khỏi tập đặc trưng đầu vào.
- **Tính toán chuẩn xác:** Làm tròn số thập phân 2 chữ số (`.round(2)`) phù hợp với bảng điểm học sinh phổ thông Việt Nam.

In [1]:
# CELL 1: Import thư viện và thiết lập cấu hình đường dẫn
import os
import gc
import pandas as pd
import numpy as np

# Xác định PROJECT_ROOT linh hoạt
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

# Định vị đường dẫn dữ liệu
RAW_DIR_S2 = os.path.join(PROJECT_ROOT, 'data', 'raw', 'S2')
RAW_FILE = os.path.join(RAW_DIR_S2, 'StudentPerformanceFactors.csv')
if not os.path.exists(RAW_FILE):
    RAW_FILE = os.path.join(PROJECT_ROOT, 'data', 'raw', 'StudentPerformanceFactors.csv')

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"✅ Đã cấu hình thư mục thành công.")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   RAW_FILE:     {RAW_FILE}")
print(f"   PROCESSED_DIR:{PROCESSED_DIR}")

✅ Đã cấu hình thư mục thành công.
   PROJECT_ROOT: D:\What-If Simulator
   RAW_FILE:     D:\What-If Simulator\data\raw\S2\StudentPerformanceFactors.csv
   PROCESSED_DIR:D:\What-If Simulator\data\processed


In [2]:
# CELL 2: Nạp dữ liệu thô và kiểm tra hiện trạng khuyết thiếu ban đầu
df_raw = pd.read_csv(RAW_FILE)
print(f"📊 Dữ liệu thô ban đầu: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột.\n")

# Thống kê các cột có giá trị khuyết thiếu (NaN)
null_counts = df_raw.isnull().sum()
missing_cols = null_counts[null_counts > 0]
print("🔍 Các cột có giá trị khuyết thiếu (Missing Values):")
for col, cnt in missing_cols.items():
    pct = (cnt / len(df_raw)) * 100
    print(f"   - {col:25s}: {cnt:4d} dòng ({pct:.2f}%)")

print("\nThống kê điểm số thô ban đầu (thang 100):")
print(f"   - Previous_Scores: min = {df_raw['Previous_Scores'].min()}, max = {df_raw['Previous_Scores'].max()}, mean = {df_raw['Previous_Scores'].mean():.2f}")
print(f"   - Exam_Score:      min = {df_raw['Exam_Score'].min()}, max = {df_raw['Exam_Score'].max()}, mean = {df_raw['Exam_Score'].mean():.2f}")

📊 Dữ liệu thô ban đầu: 6,607 dòng x 20 cột.

🔍 Các cột có giá trị khuyết thiếu (Missing Values):
   - Teacher_Quality          :   78 dòng (1.18%)
   - Parental_Education_Level :   90 dòng (1.36%)
   - Distance_from_Home       :   67 dòng (1.01%)

Thống kê điểm số thô ban đầu (thang 100):
   - Previous_Scores: min = 50, max = 100, mean = 75.07
   - Exam_Score:      min = 55, max = 101, mean = 67.24


In [3]:
# CELL 3: Sửa lỗi ngoại lai (Bug Fix) & Quy đổi đồng bộ thang điểm 10
df_clean = df_raw.copy()

# 1. Phát hiện và sửa lỗi ngoại lai điểm thi vượt trần 100
outlier_mask = df_clean['Exam_Score'] > 100
outlier_count = outlier_mask.sum()
print(f"⚠️ Phát hiện {outlier_count} bản ghi có Exam_Score > 100 (Ví dụ: dòng có điểm 101).")
df_clean['Exam_Score'] = df_clean['Exam_Score'].clip(upper=100)
print("🛡️ [BUG FIX] Đã clip toàn bộ điểm thi về mức trần tối đa hợp lệ là 100.")

# 2. Quy đổi về thang điểm 10 chuẩn giáo dục Việt Nam
df_clean['Previous_Scores'] = (df_clean['Previous_Scores'] / 10.0).round(2)
df_clean['Exam_Score']      = (df_clean['Exam_Score']      / 10.0).round(2)

# 3. Kiểm tra và giới hạn thời lượng ngủ sinh lý hợp lý [4, 10] giờ/đêm
df_clean['Sleep_Hours'] = df_clean['Sleep_Hours'].clip(lower=4, upper=10)

print(f"✅ Thang điểm sau quy đổi (thang 10):")
print(f"   - Previous_Scores: min = {df_clean['Previous_Scores'].min():.2f}, max = {df_clean['Previous_Scores'].max():.2f}, mean = {df_clean['Previous_Scores'].mean():.2f}")
print(f"   - Exam_Score:      min = {df_clean['Exam_Score'].min():.2f}, max = {df_clean['Exam_Score'].max():.2f}, mean = {df_clean['Exam_Score'].mean():.2f}")
print(f"   - Sleep_Hours:      min = {df_clean['Sleep_Hours'].min():.1f}, max = {df_clean['Sleep_Hours'].max():.1f}")

⚠️ Phát hiện 1 bản ghi có Exam_Score > 100 (Ví dụ: dòng có điểm 101).


🛡️ [BUG FIX] Đã clip toàn bộ điểm thi về mức trần tối đa hợp lệ là 100.
✅ Thang điểm sau quy đổi (thang 10):
   - Previous_Scores: min = 5.00, max = 10.00, mean = 7.51
   - Exam_Score:      min = 5.50, max = 10.00, mean = 6.72
   - Sleep_Hours:      min = 4.0, max = 10.0


In [4]:
# CELL 4: Xử lý giá trị khuyết thiếu bằng Mode (Giá trị xuất hiện phổ biến nhất)
categorical_missing = ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']

print("🛠️ Tiến hành điền khuyết thiếu bằng Mode cho các biến phân loại:")
for col in categorical_missing:
    if col in df_clean.columns:
        mode_val = df_clean[col].mode()[0]
        missing_count = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f"   - Điền mode '{mode_val}' cho {missing_count} dòng khuyết thiếu ở cột '{col}'.")

# Kiểm tra đảm bảo không còn giá trị null nào tồn tại
assert df_clean.isna().sum().sum() == 0, "❌ Lỗi: Vẫn còn giá trị null sau khi điền!"
print("\n✅ Hoàn tất xử lý Missing Values: 100% dữ liệu đã sạch (0 giá trị null).")

🛠️ Tiến hành điền khuyết thiếu bằng Mode cho các biến phân loại:
   - Điền mode 'Medium' cho 78 dòng khuyết thiếu ở cột 'Teacher_Quality'.
   - Điền mode 'High School' cho 90 dòng khuyết thiếu ở cột 'Parental_Education_Level'.
   - Điền mode 'Near' cho 67 dòng khuyết thiếu ở cột 'Distance_from_Home'.

✅ Hoàn tất xử lý Missing Values: 100% dữ liệu đã sạch (0 giá trị null).


In [5]:
# CELL 5: Xây dựng đặc trưng tương tác học tập & Tạo nhãn mục tiêu at_risk

# 1. Đặc trưng tương tác học tập: effective_study_index
# Phản ánh tích lũy thời gian học tập thực chất (giờ học * tỷ lệ chuyên cần)
df_clean['effective_study_index'] = (
    df_clean['Hours_Studied'] * (df_clean['Attendance'] / 100.0)
).round(2)

# 2. Tạo nhãn mục tiêu Tầng 1 (Phân loại nguy cơ học tập)
# Ngưỡng 6.5: Học sinh có điểm thi < 6.5 (Dưới loại Khá chuẩn đầu ra THPT)
RISK_THRESHOLD = 6.5
df_clean['at_risk'] = (df_clean['Exam_Score'] < RISK_THRESHOLD).astype(int)

# Thống kê phân bố nhãn
risk_counts = df_clean['at_risk'].value_counts()
risk_pct = df_clean['at_risk'].mean() * 100

print(f"✅ Đã tạo đặc trưng 'effective_study_index' và nhãn 'at_risk' (ngưỡng < {RISK_THRESHOLD}):")
print(f"   - Số học sinh An toàn (Safe, at_risk = 0):     {risk_counts.get(0, 0):4d} ({100-risk_pct:.2f}%)")
print(f"   - Số học sinh Nguy cơ (At-Risk, at_risk = 1):   {risk_counts.get(1, 0):4d} ({risk_pct:.2f}%)")
print(f"   - Tỷ lệ học sinh cần can thiệp:                {risk_pct:.2f}%")

✅ Đã tạo đặc trưng 'effective_study_index' và nhãn 'at_risk' (ngưỡng < 6.5):
   - Số học sinh An toàn (Safe, at_risk = 0):     5155 (78.02%)
   - Số học sinh Nguy cơ (At-Risk, at_risk = 1):   1452 (21.98%)
   - Tỷ lệ học sinh cần can thiệp:                21.98%


In [6]:
# CELL 6: Xuất bảng Master sạch duy nhất & Kiểm tra Checkpoints nghiêm ngặt
output_master_path = os.path.join(PROCESSED_DIR, 'student_performance_master.csv')
df_clean.to_csv(output_master_path, index=False)
print(f"💾 Đã lưu thành công Master Dataset: {output_master_path}")

# ============================================================
# CHECKPOINTS KIỂM ĐỊNH CHẤT LƯỢNG (ACCEPTANCE CRITERIA)
# ============================================================
assert len(df_clean) == 6607, f"❌ Lỗi: Số lượng dòng không khớp 6,607 (Thực tế: {len(df_clean)})!"
assert df_clean['Exam_Score'].between(0.0, 10.0).all(), "❌ Lỗi: Exam_Score ngoài phạm vi [0.0, 10.0]!"
assert df_clean['Previous_Scores'].between(0.0, 10.0).all(), "❌ Lỗi: Previous_Scores ngoài phạm vi [0.0, 10.0]!"
assert df_clean['Sleep_Hours'].between(4.0, 10.0).all(), "❌ Lỗi: Sleep_Hours ngoài phạm vi [4.0, 10.0]!"
assert df_clean['at_risk'].isin([0, 1]).all(), "❌ Lỗi: at_risk chứa giá trị không hợp lệ!"
assert df_clean.isna().sum().sum() == 0, "❌ Lỗi: Còn giá trị null trong bảng Master!"

# Kiểm tra các cột chính có mặt đầy đủ
EXPECTED_COLUMNS = [
    'Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources',
    'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores',
    'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income',
    'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity',
    'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home',
    'Gender', 'Exam_Score', 'effective_study_index', 'at_risk'
]
assert all(c in df_clean.columns for c in EXPECTED_COLUMNS), "❌ Lỗi: Thiếu cột đặc trưng trong bảng Master!"

print("\n" + "="*65)
print("🎉 TASK 1.1 CHECKPOINTS: TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN XUẤT SẮC!")
print(f"   Tổng số bản ghi Master: {len(df_clean):,} học sinh")
print(f"   Tổng số cột đặc trưng:  {df_clean.shape[1]} cột (bao gồm effective_study_index và at_risk)")
print(f"   Thang điểm Exam_Score:  min={df_clean['Exam_Score'].min():.2f}, max={df_clean['Exam_Score'].max():.2f}, mean={df_clean['Exam_Score'].mean():.2f}/10")
print(f"   Tỷ lệ nhãn nguy cơ:     {risk_pct:.2f}% (Bài toán phân loại cân bằng hợp lý)")
print("="*65)
print("\n🏆 CHÚC MỪNG BẠN ĐÃ HOÀN THÀNH 100% TASK 1.1! 🏆")

💾 Đã lưu thành công Master Dataset: D:\What-If Simulator\data\processed\student_performance_master.csv

🎉 TASK 1.1 CHECKPOINTS: TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN XUẤT SẮC!
   Tổng số bản ghi Master: 6,607 học sinh
   Tổng số cột đặc trưng:  22 cột (bao gồm effective_study_index và at_risk)
   Thang điểm Exam_Score:  min=5.50, max=10.00, mean=6.72/10
   Tỷ lệ nhãn nguy cơ:     21.98% (Bài toán phân loại cân bằng hợp lý)

🏆 CHÚC MỪNG BẠN ĐÃ HOÀN THÀNH 100% TASK 1.1! 🏆
